# Runiverse · ACE-Step 1.5 추론 시작하기
**Google Colab / 추론용 Python 3.11 / NVIDIA CUDA GPU**

런타임 메뉴에서 **런타임 유형 변경 → GPU**를 선택한 뒤 위에서 아래로 실행하세요.
기본 실험은 **Turbo · 30초 · 170 BPM · 무보컬 · 1개 · 5Hz LM caption 개선**입니다. 4-4까지 실행하면 카드 전체(기본 30초 + 사진별 10초)를 만듭니다.

**Python 환경:** 코랩 노트북 커널을 강제로 바꾸지 않습니다. `uv`가 별도 Python 3.11과 가상환경을 만들고, 다운로드·모델 로딩·추론은 모두 그 환경의 Python으로 실행합니다. 코랩 커널은 설정·결과 표시만 담당합니다. 설치 후 런타임을 재시작할 필요가 없습니다.

처음에는 라이브러리와 수 GB의 모델 다운로드가 필요합니다. 코랩 GPU 할당·세션 유지가 보장되는 것은 아닙니다. 결과를 다운로드하거나 마지막 선택 셀로 Drive에 복사하세요.

**검증 범위:** 공식 소스의 함수·매개변수와 대조하고 노트북 구조·문법을 검증한 초안입니다. 실제 코랩 GPU에서 설치·음원 생성은 아직 실행하지 않았습니다.


## 1. GPU와 작업 경로 확인
CUDA가 표시된 `nvidia-smi` 출력은 드라이버 지원 정보를 뜻합니다. 설치 후 PyTorch에서 실제 CUDA 연산을 한 번 더 확인합니다.


In [ ]:
import sys
import os
import json
import subprocess
import shutil
from pathlib import Path

if not Path("/content").exists():
    raise RuntimeError("이 노트북은 Google Colab의 /content 환경을 기준으로 작성했습니다.")

ROOT = Path("/content/runiverse_acestep")
REPO = ROOT / "ACE-Step-1.5"
OUTPUT_ROOT = ROOT / "outputs"
ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# 확인 당시 공식 main의 커밋. 실험 재현을 위해 고정합니다.
SOURCE_COMMIT = "ca1e85fe9430179831e6bc6be790c332190a3866"
REPO_URL = "https://github.com/ace-step/ACE-Step-1.5.git"

print("코랩 커널 Python:", sys.version.split()[0])
print("추론용 Python은 다음 셀에서 별도 3.11로 설치합니다.")
try:
    gpu_check = subprocess.run(["nvidia-smi"], check=True, text=True, capture_output=True)
except (FileNotFoundError, subprocess.CalledProcessError) as exc:
    raise RuntimeError("GPU 런타임을 선택하고 다시 실행하세요.") from exc
print(gpu_check.stdout)
print(f"남은 디스크: {shutil.disk_usage('/content').free / 2**30:.1f} GiB")


## 2. Python 3.11과 공식 의존성 설치
공식 저장소의 `uv.lock`을 `--frozen`으로 사용합니다. 별도 환경이라 코랩 기본 PyTorch를 덮어쓰지 않습니다.
고정 소스는 Linux x86_64용 PyTorch 2.10.0 / CUDA 12.8 패키지를 지정합니다. 코랩 드라이버가 호환되어야 합니다.
설치 오류가 나면 로그를 확인하세요. 중간에 실패해도 이 셀을 다시 실행할 수 있습니다.


In [ ]:
def run_command(args, *, cwd=None, env=None):
    # 문자열 shell 대신 인자 배열로 실행합니다. 설치 로그는 바로 표시됩니다.
    print("실행:", " ".join(map(str, args)))
    subprocess.run(list(map(str, args)), cwd=cwd, env=env, check=True)

run_command([sys.executable, "-m", "pip", "install", "--quiet", "uv>=0.8"])
UV = shutil.which("uv")
if not UV:
    raise RuntimeError("uv 실행 파일을 찾지 못했습니다.")

run_command([UV, "python", "install", "3.11"])

if not REPO.exists():
    run_command(["git", "clone", "--no-checkout", "--depth", "1", REPO_URL, REPO])
elif not (REPO / ".git").exists():
    raise RuntimeError(f"{REPO}에 다른 파일이 있습니다. 별도 작업 경로를 사용하세요.")

run_command(["git", "fetch", "--depth", "1", "origin", SOURCE_COMMIT], cwd=REPO)
run_command(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=REPO)
run_command([UV, "sync", "--frozen", "--no-dev", "--python", "3.11"], cwd=REPO)

PY311 = REPO / ".venv" / "bin" / "python"
if not PY311.exists():
    raise RuntimeError("추론용 Python 환경 생성에 실패했습니다.")

INFER_ENV = os.environ.copy()
INFER_ENV.update({
    "TOKENIZERS_PARALLELISM": "false",
    "ACESTEP_CHECKPOINTS_DIR": str(REPO / "checkpoints"),
    "HF_HOME": str(ROOT / "hf_cache"),
})

def run_py311(script, *args):
    run_command([PY311, script, *args], cwd=REPO, env=INFER_ENV)

print("추론용 Python:", PY311)

In [ ]:
INFER_ENV["MPLBACKEND"] = "Agg"

## 3. Python 3.11 및 실제 CUDA 연산 확인
GPU가 없으면 CPU로 조용히 전환하지 않고 중단합니다. T4처럼 BF16을 지원하지 않는 GPU는 ACE-Step이 FP16을 선택합니다.


In [ ]:
gpu_probe = ROOT / "gpu_probe.py"
gpu_probe.write_text("""
import json
import platform
import sys
import torch
from pathlib import Path

assert sys.version_info[:2] == (3, 11), sys.version
if not torch.cuda.is_available():
    raise RuntimeError("PyTorch에서 CUDA를 사용할 수 없습니다. GPU 런타임과 드라이버 호환성을 확인하세요.")
x = torch.ones((32, 32), device="cuda")
y = x @ x
torch.cuda.synchronize()
assert y.device.type == "cuda" and float(y[0, 0]) == 32.0
p = torch.cuda.get_device_properties(0)
info = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "gpu": p.name,
    "gpu_vram_gib": round(p.total_memory / 2**30, 2),
    "bf16_supported": bool(torch.cuda.is_bf16_supported()),
    "compute_capability": list(torch.cuda.get_device_capability(0)),
}
Path(sys.argv[1]).write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(info, ensure_ascii=False, indent=2))
""", encoding="utf-8")
run_py311(gpu_probe, ROOT / "gpu_info.json")
GPU_INFO = json.loads((ROOT / "gpu_info.json").read_text(encoding="utf-8"))
if GPU_INFO["gpu_vram_gib"] < 12:
    print("VRAM이 작습니다. 우선 10~30초, batch_size=1, LM 미사용을 유지하세요.")


## 4. 첫 실험 설정
`caption`은 음악의 장르·악기·분위기 설명입니다. BPM·길이는 별도 조건입니다.
BPM은 **목표값**이며 실제 음원의 BPM 측정 결과가 아닙니다. 같은 seed도 GPU·라이브러리가 달라지면 완전히 동일한 결과를 보장하지 않습니다.

`use_lm=True`는 5Hz LM으로 caption만 개선합니다 (`use_cot_caption=True`, 오디오 코드 생성 `thinking=False`, 전처리 모듈 문서 7장). LM은 메인 모델 묶음에 들어 있는 `acestep-5Hz-lm-1.7B`(서빙과 동일)이라 추가 다운로드가 없습니다.
VRAM이 부족하면 `use_lm=False`로 LM 없이 비교하세요. `offload_to_cpu=True`는 일부 모델을 CPU에 두어 VRAM을 줄이는 설정이며 시스템 RAM과 전송 시간이 더 필요할 수 있습니다.

In [ ]:
EXPERIMENT = {
    "caption": (
        "Energetic electronic running music, steady clear drum beat, "
        "uplifting synth melody, consistent rhythm, instrumental, no vocals"
    ),
    "bpm": 170,
    "duration": 30.0,
    "seed": 42,
    "instrumental": True,
    "inference_steps": 8,
    "batch_size": 1,
    "audio_format": "wav",
    "use_lm": True,  # 5Hz LM caption 개선 (문서 7장 1차 설정)
    "offload_to_cpu": True,
}
assert 30 <= EXPERIMENT["bpm"] <= 300
assert 10 <= EXPERIMENT["duration"] <= 600
assert EXPERIMENT["batch_size"] == 1, "초안에서는 1개씩 생성하세요."
print(json.dumps(EXPERIMENT, ensure_ascii=False, indent=2))

## 4-1. 전처리·분석 코드 (`src` 사본)
카드 1건의 공통 설정(BPM·조성·박자), Qwen3-VL 분석, 곡별 `caption_input`을 만듭니다. 4-1~4-2는 GPU·모델 없이, 4-3~4-4는 GPU로 실제 Qwen3-VL을 돌립니다.

- 아래 `%%writefile` 셀 9개는 저장소 `src/runiverse_music/` 파일을 **그대로 복사**한 것입니다 → 서빙 코드와 같은 규칙·프롬프트
- **여기서 고치지 마세요.** `src`를 고친 뒤 `python scripts/sync_notebook.py`로 다시 복사합니다 (`tests/test_notebook_sync.py`가 일치 검사)
- 규칙값·프롬프트 실험은 4-2·4-4 셀에서 `rules.DEFAULT_BPM = 150`, `analyzer.PHOTO_PROMPT = "..."`처럼 바꿉니다. import 셀을 다시 실행하면 원래 값으로 돌아갑니다
- 런타임 연결 후 4-1~4-2만 실행해도 됩니다 (1~3번 설치 셀 불필요)
- 6~7번 음악 생성도 이 사본(`generator.AceStepGenerator`)을 씁니다 → **생성 전에 4-1은 꼭 실행**

In [ ]:
# 4-1. src 사본을 둘 패키지 폴더. 이어지는 %%writefile 셀이 이 폴더에 파일을 씁니다.
from pathlib import Path

PKG = Path("/content/rv_src/runiverse_music")
PKG.mkdir(parents=True, exist_ok=True)
(PKG / "__init__.py").write_text("")

In [ ]:
%%writefile /content/rv_src/runiverse_music/mapping_rules.py
"""전처리 규칙값 (v1). 로직 없이 값만 둔다 — 계산은 music_params.py.

정본: docs/AI_음악_전처리_모듈.md 4장 (Notion "Ace-step 메타데이터 (공통 파라미터)")
값을 바꾸면: MAPPING_VERSION 올리기 → tests/test_music_params.py 기대값 수정 → 이미지 재빌드
실험만 할 때: pytest는 monkeypatch, Colab은 generator에 bpm·keyscale·duration을 직접 전달
"""

# 결과 manifest에 기록해 "어떤 규칙표로 만든 음악인지" 추적한다
MAPPING_VERSION = "v1"

# ── 페이스 → BPM ─────────────────────────────────────────────
# (상한 초/km, BPM). 위에서부터 차례로 비교해 "상한 미만"인 첫 구간을 고른다
PACE_TO_BPM: tuple[tuple[int, int], ...] = (
    (270, 180),  # 4:30/km 미만
    (360, 170),  # 4:30 ~ 5:59/km
    (450, 160),  # 6:00 ~ 7:29/km
)
SLOWEST_BPM = 150  # 7:30/km 이상
DEFAULT_BPM = 160  # 페이스 결측·유효하지 않음

# ── WMO Code Table 4677 → 날씨 그룹 ──────────────────────────
# (그룹, ((시작, 끝), ...)). 시작·끝 모두 포함. 0~99를 빈칸·중복 없이 덮는다
WEATHER_RANGES: tuple[tuple[str, tuple[tuple[int, int], ...]], ...] = (
    ("DRY_OR_RECENT", ((0, 3), (14, 16), (20, 29))),
    ("HAZE_OR_FOG", ((4, 12), (30, 35), (40, 49))),
    ("RAIN", ((50, 67), (80, 82), (91, 92))),
    ("SNOW_OR_ICE", ((36, 39), (68, 79), (83, 90), (93, 94))),
    ("STORM_OR_WIND", ((13, 13), (17, 19), (95, 99))),
)
# 가정값: 문서상 "기본 날씨 그룹" 미정 → 현재 현상이 없는 그룹으로 둔다 (협의 후 변경)
DEFAULT_WEATHER_GROUP = "DRY_OR_RECENT"

# ── 회고 방향 × 날씨 그룹 → 조성(keyscale) ────────────────────
# 열 순서 = WEATHER_GROUPS. 행 = Qwen3-VL이 고르는 emotion_direction 7종
WEATHER_GROUPS = ("DRY_OR_RECENT", "HAZE_OR_FOG", "RAIN", "SNOW_OR_ICE", "STORM_OR_WIND")
KEYSCALE_TABLE: dict[str, tuple[str, str, str, str, str]] = {
    "평온·담담함": ("C Major", "F Major", "F Major", "Eb Major", "C Major"),
    "설렘·가벼운 즐거움": ("A Major", "A Major", "D Major", "Eb Major", "D Major"),
    "성취·환희": ("D Major", "D Major", "E Major", "Eb Major", "E Major"),
    "감사·따뜻한 위로": ("G Major", "F Major", "G Major", "Eb Major", "Eb Major"),
    "아쉬움·그리움": ("G Minor", "E Minor", "G Minor", "D Minor", "C Minor"),
    "고뇌·결연함": ("C Minor", "F# Minor", "C Minor", "D Minor", "C Minor"),
    "깊은 슬픔·상실감": ("A Minor", "E Minor", "D Minor", "F Minor", "F Minor"),
}
# analyzer가 Qwen3-VL 출력을 검증할 때도 이 목록을 쓴다
EMOTION_DIRECTIONS: tuple[str, ...] = tuple(KEYSCALE_TABLE)
DEFAULT_EMOTION_DIRECTION = "평온·담담함"  # 회고 해석 실패·판단 불가

# ── 박자·길이 ────────────────────────────────────────────────
TIME_SIGNATURE = "4"           # 4/4박자. ACE-Step은 분자만 문자열로 받는다
BASE_DURATION_SECONDS = 30.0   # 기본 음악
PHOTO_DURATION_SECONDS = 10.0  # 사진별 음악

# ── 회고 분석 실패 시 기본값 ─────────────────────────────────
DEFAULT_EMOTION_CUE = "담담하고 안정적인 분위기"
DEFAULT_MUSIC_CUE_EN = "a calm and steady mood"  # 가정값: 문서는 "담담한 분위기의 기본값"까지만 정함

# ── caption_input ────────────────────────────────────────────
# 활동유형 → 영어 활동 맥락. TRAVEL은 문서 예시, DAILY·RACE는 가정값
ACTIVITY_CONTEXT: dict[str, str] = {
    "DAILY": "an everyday run",
    "TRAVEL": "a run while traveling",
    "RACE": "a race-day run",
}
CAPTION_MAX_CHARS = 512

# ── 생성 seed ────────────────────────────────────────────────
# 카드 입력에 seed가 없어 고정값을 쓴다 (가정값). 사진 곡은 seed + (사진 순서 + 1)로 파생
DEFAULT_SEED = 42


In [ ]:
%%writefile /content/rv_src/runiverse_music/music_params.py
"""카드 공통 음악 수치(BPM·조성·박자) 결정. 모델 호출 없이 규칙표만 조회한다.

규칙값은 mapping_rules.py에 있다. 모듈째 import해 호출 시점에 값을 읽으므로
mapping_rules 값을 바꾸면(monkeypatch 등) 결과에 바로 반영된다.
"""
from dataclasses import asdict, dataclass

from runiverse_music import mapping_rules as rules


def bpm_from_pace(avg_pace) -> int:
    """평균 페이스(초/km) → 카드 공통 BPM.

    정수가 아니거나 0 이하이면 페이스를 알 수 없다고 보고 DEFAULT_BPM을 쓴다.
    """
    # bool은 int의 하위 타입이라 True가 1초로 통과하지 않도록 먼저 걸러 낸다
    if isinstance(avg_pace, bool) or not isinstance(avg_pace, int) or avg_pace <= 0:
        return rules.DEFAULT_BPM
    for upper, bpm in rules.PACE_TO_BPM:
        if avg_pace < upper:
            return bpm
    return rules.SLOWEST_BPM


def weather_group(weather_code) -> str:
    """WMO ww 코드(0~99) → 조성표의 날씨 그룹.

    코드가 없거나 범위 밖이면 DEFAULT_WEATHER_GROUP. 실제 날씨를 지어내지는 않는다
    (그룹은 조성 선택에만 쓰고, caption에 날씨를 쓰지 않는다).
    """
    if isinstance(weather_code, bool) or not isinstance(weather_code, int):
        return rules.DEFAULT_WEATHER_GROUP
    for group, ranges in rules.WEATHER_RANGES:
        if any(start <= weather_code <= end for start, end in ranges):
            return group
    return rules.DEFAULT_WEATHER_GROUP  # 0 미만·99 초과


def keyscale_for(emotion_direction, group) -> str:
    """emotion_direction × 날씨 그룹 → ACE-Step keyscale (예: "E Major").

    목록에 없는 방향은 DEFAULT_EMOTION_DIRECTION, 없는 그룹은 DEFAULT_WEATHER_GROUP으로 본다.
    """
    table = rules.KEYSCALE_TABLE
    row = table.get(emotion_direction) or table[rules.DEFAULT_EMOTION_DIRECTION]
    if group not in rules.WEATHER_GROUPS:
        group = rules.DEFAULT_WEATHER_GROUP
    return row[rules.WEATHER_GROUPS.index(group)]


@dataclass(frozen=True)
class SharedParams:
    """카드 안 모든 곡(기본·사진별)에 똑같이 넘기는 ACE-Step 파라미터.

    곡마다 달라지는 값(duration·caption_input)은 여기 두지 않는다.
    """

    bpm: int
    keyscale: str
    timesignature: str
    instrumental: bool = True

    def to_dict(self) -> dict[str, object]:
        """응답·analysis.json의 "shared" 형식."""
        return asdict(self)


def decide_shared(*, avg_pace, weather_code, emotion_direction) -> SharedParams:
    """러닝 기록 + 회고 분석 결과 → 카드 공통 설정.

    emotion_direction은 회고 분석(②) 결과라서, 이 함수는 반드시 회고 분석 뒤에 부른다.
    """
    return SharedParams(
        bpm=bpm_from_pace(avg_pace),
        keyscale=keyscale_for(emotion_direction, weather_group(weather_code)),
        timesignature=rules.TIME_SIGNATURE,
    )


In [ ]:
%%writefile /content/rv_src/runiverse_music/prompting.py
"""곡별 caption_input 조립. 정본: docs/AI_음악_전처리_모듈.md 6장.

BPM·조성·박자·길이·장르·악기는 넣지 않는다 — 수치는 별도 파라미터로 넘긴다.
"""

from typing import Callable

from runiverse_music import mapping_rules as rules


def build_caption_input(activity: str, cue: str | None) -> str:
    """기본 음악: 활동 맥락 + 회고 단서(music_cue_en). cue가 없으면 활동 맥락만 쓴다."""
    head = f"An instrumental piece reflecting {rules.ACTIVITY_CONTEXT[activity]}"
    return _fit(cue, lambda text: f"{head} and {text}.", f"{head}.")


def build_photo_caption(activity: str, cue: str | None) -> str:
    """사진 음악: 사진 단서를 맨 앞에 두어 곡끼리 구분되게 한다. 활동 맥락은 뒤에 짧게 유지.

    cue가 없으면(사진 분석 실패) 기본 음악과 같은 활동 맥락 문장을 쓴다.
    """
    tail = f", an instrumental piece from {rules.ACTIVITY_CONTEXT[activity]}."
    return _fit(cue, lambda text: f"{text[0].upper()}{text[1:]}{tail}", build_caption_input(activity, None))


def _fit(cue: str | None, render: Callable[[str], str], fallback: str) -> str:
    """512자를 넘으면 cue를 뒤에서부터 단어 단위로 줄인다 — 활동 맥락은 항상 보존."""
    words = (cue or "").strip().rstrip(".").split()
    while words:
        caption = render(" ".join(words))
        if len(caption) <= rules.CAPTION_MAX_CHARS:
            return caption
        words.pop()
    return fallback


In [ ]:
%%writefile /content/rv_src/runiverse_music/schemas.py
"""카드 입력 형식. 정본: docs/AI_음악_전처리_모듈.md 1장."""

from pydantic import BaseModel, ConfigDict, Field, field_validator

from runiverse_music import mapping_rules as rules


class PhotoInput(BaseModel):
    """사진 1장. 별도 ID가 없어 photos 배열 순서(0부터)로 결과와 연결한다."""

    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)

    image_key: str = Field(min_length=1)  # 분석할 원본 사진의 S3 키
    caption: str | None = None            # 사용자 사진 설명


class RunningSummary(BaseModel):
    # 거리·케이던스 등 나머지 지표는 현재 매핑에 쓰지 않아 받기만 하고 버린다
    model_config = ConfigDict(extra="ignore")

    avg_pace: int | None = None      # 초/km. 결측·0 이하는 music_params가 기본 BPM으로 처리
    weather_code: int | None = None  # WMO 4677 ww


class CardRequest(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)

    user_id: int
    record_card_id: int
    memo: str | None = None  # 회고 글
    activity_type: str
    running_summary: RunningSummary
    photos: list[PhotoInput] = []

    @field_validator("activity_type")
    @classmethod
    def _known_activity(cls, value: str) -> str:
        if value not in rules.ACTIVITY_CONTEXT:
            raise ValueError(f"activity_type은 {', '.join(rules.ACTIVITY_CONTEXT)} 중 하나여야 합니다.")
        return value


In [ ]:
%%writefile /content/rv_src/runiverse_music/config.py
"""컨테이너 환경변수 → Settings. 잘못된 값은 시작 시 ConfigError."""
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Mapping


class ConfigError(ValueError):
    """환경변수 값이 잘못됨."""


@dataclass(frozen=True)
class Settings:
    analyzer: str
    generator: str
    storage: str
    output_bucket: str | None
    photo_bucket: str | None  # 사진 원본을 읽는 기존 사진 버킷 (AI 버킷과 별도)
    output_prefix: str
    local_output_dir: Path | None
    inference_steps: int
    offload_to_cpu: bool
    use_lm: bool  # 5Hz LM으로 caption 개선 (use_cot_caption)
    audio_format: str
    log_level: str
    model_dir: Path
    acestep_root: Path
    acestep_commit: str
    image_tag: str


def _choice(env: Mapping[str, str], key: str, default: str, choices: tuple[str, ...]) -> str:
    value = env.get(key, default).strip().lower()
    if value not in choices:
        raise ConfigError(f"{key}={value!r}: {', '.join(choices)} 중 하나여야 합니다.")
    return value


def _flag(env: Mapping[str, str], key: str, default: str) -> bool:
    return _choice(env, key, default, ("true", "false")) == "true"


def _number(env: Mapping[str, str], key: str, default: str, cast: Callable, low: float, high: float):
    raw = env.get(key, default).strip()
    try:
        value = cast(raw)
    except ValueError:
        raise ConfigError(f"{key}={raw!r}: 숫자여야 합니다.") from None
    if not low <= value <= high:  # NaN도 여기서 걸림
        raise ConfigError(f"{key}={raw!r}: {low:g}~{high:g} 범위여야 합니다.")
    return value


def load_settings(env: Mapping[str, str] | None = None) -> Settings:
    env = os.environ if env is None else env

    storage = _choice(env, "STORAGE", "s3", ("s3", "local"))
    bucket = env.get("OUTPUT_BUCKET", "").strip() or None
    if storage == "s3" and bucket is None:
        raise ConfigError("STORAGE=s3이면 OUTPUT_BUCKET이 필요합니다.")
    photo_bucket = env.get("PHOTO_BUCKET", "").strip() or None
    if storage == "s3" and photo_bucket is None:
        raise ConfigError("STORAGE=s3이면 PHOTO_BUCKET이 필요합니다.")
    local_dir = env.get("LOCAL_OUTPUT_DIR", "").strip()
    if storage == "local" and not local_dir:
        raise ConfigError("STORAGE=local이면 LOCAL_OUTPUT_DIR이 필요합니다.")
    prefix = env.get("OUTPUT_PREFIX", "music/outputs").strip().strip("/")
    if not prefix:
        raise ConfigError("OUTPUT_PREFIX가 비어 있습니다.")

    return Settings(
        analyzer=_choice(env, "ANALYZER", "qwen", ("qwen", "fake")),
        generator=_choice(env, "GENERATOR", "acestep", ("acestep", "fake")),
        storage=storage,
        output_bucket=bucket,
        photo_bucket=photo_bucket,
        output_prefix=prefix,
        local_output_dir=Path(local_dir) if local_dir else None,
        inference_steps=_number(env, "INFERENCE_STEPS", "8", int, 1, 200),
        offload_to_cpu=_flag(env, "OFFLOAD_TO_CPU", "true"),
        use_lm=_flag(env, "USE_LM", "true"),
        audio_format=_choice(env, "AUDIO_FORMAT", "wav", ("wav",)),
        log_level=_choice(env, "LOG_LEVEL", "INFO", ("debug", "info", "warning", "error")).upper(),
        model_dir=Path(env.get("MODEL_DIR", "/opt/ml/model")),
        acestep_root=Path(env.get("ACESTEP_ROOT", "/opt/ace-step")),
        acestep_commit=env.get("ACESTEP_COMMIT", "unknown"),
        image_tag=env.get("IMAGE_TAG", "unknown"),
    )


In [ ]:
%%writefile /content/rv_src/runiverse_music/generator.py
"""음악 생성기. torch·acestep·soundfile은 AceStepGenerator 메서드 안에서만 import한다."""
from __future__ import annotations

import hashlib
import json
import os
import sys
import time
import wave
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Protocol

from .config import Settings

CONFIG_PATH = "acestep-v15-turbo"
LM_MODEL = "acestep-5Hz-lm-1.7B"  # Ace-Step1.5 모델 묶음(model.tar.gz)에 함께 들어 있는 LM


class GenerationError(RuntimeError):
    """생성 실패 (요청 단위)."""


class CudaOutOfMemory(GenerationError):
    """GPU 메모리 부족."""


@dataclass(frozen=True)
class MusicJob:
    """곡 1개 생성 입력. 수치는 caption에 쓰지 않고 ACE-Step 파라미터로 따로 넘긴다."""

    caption: str
    bpm: int
    keyscale: str       # "" → ACE-Step이 정함 (노트북 EXPERIMENT)
    timesignature: str
    duration: float
    seed: int


@dataclass(frozen=True)
class GenerationResult:
    audio_path: Path
    sample_rate: int
    channels: int
    actual_duration_seconds: float
    size_bytes: int
    generation_seconds: float
    peak_torch_allocated_gib: float
    lm_caption: str | None  # 5Hz LM이 개선한 caption. LM을 안 쓰면 None


class Generator(Protocol):
    manifest: dict[str, Any]

    def load(self) -> None: ...

    def generate(self, job: MusicJob, out_dir: Path) -> GenerationResult: ...


def check_audio_file(path: Path) -> Path:
    # 성공 플래그가 있어도 파일 저장이 실패할 수 있어 별도로 검사한다.
    if not path.is_file() or path.stat().st_size == 0:
        raise GenerationError(f"음원 파일이 없거나 비어 있습니다: {path}")
    return path


class FakeGenerator:
    """모델 없이 1초 무음 wav를 만든다. 로컬·컨테이너 계약 검증용."""

    SAMPLE_RATE = 48000
    CHANNELS = 2

    def __init__(self) -> None:
        self.manifest: dict[str, Any] = {"generator": "fake"}

    def load(self) -> None:
        pass

    def generate(self, job: MusicJob, out_dir: Path) -> GenerationResult:
        path = Path(out_dir) / "fake.wav"
        with wave.open(str(path), "wb") as w:
            w.setnchannels(self.CHANNELS)
            w.setsampwidth(2)
            w.setframerate(self.SAMPLE_RATE)
            w.writeframes(b"\x00" * self.SAMPLE_RATE * self.CHANNELS * 2)
        check_audio_file(path)
        return GenerationResult(path, self.SAMPLE_RATE, self.CHANNELS, 1.0, path.stat().st_size, 0.0, 0.0, None)


def _read_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else {}


def _sha256(path: Path) -> str | None:
    return hashlib.sha256(path.read_bytes()).hexdigest() if path.is_file() else None


class AceStepGenerator:
    """ACE-Step 1.5 Turbo. 노트북 infer_music.py의 로딩·생성 로직."""

    def __init__(self, settings: Settings) -> None:
        self._settings = settings
        self._dit = None
        self._lm = None
        self.manifest: dict[str, Any] = {}

    def load(self) -> None:
        import torch
        from acestep.handler import AceStepHandler

        s = self._settings
        if not torch.cuda.is_available():
            raise GenerationError("CUDA GPU가 필요합니다. CPU로 전환하지 않습니다.")
        if not s.model_dir.is_dir():
            raise GenerationError(f"모델 폴더가 없습니다: {s.model_dir}")
        os.environ["ACESTEP_CHECKPOINTS_DIR"] = str(s.model_dir)

        started = time.perf_counter()
        dit = AceStepHandler()
        # 반환 순서는 공식 API의 (status_message, success)
        status, success = dit.initialize_service(
            project_root=str(s.acestep_root),
            config_path=CONFIG_PATH,
            device="cuda",
            use_flash_attention=False,
            compile_model=False,
            quantization=None,
            offload_to_cpu=s.offload_to_cpu,
            offload_dit_to_cpu=False,
            prefer_source="huggingface",
        )
        if not success:
            raise GenerationError(f"DiT 초기화 실패: {status}")
        if str(dit.device) != "cuda":
            raise GenerationError(f"CUDA 요청이 {dit.device}로 전환되었습니다.")
        if s.use_lm:
            self._lm = _load_lm(s, dit.dtype)
        torch.cuda.synchronize()
        self._dit = dit
        self.manifest = {
            "source_commit": s.acestep_commit,
            "model_revisions": _read_json(s.model_dir / "model_revisions.json"),
            "uv_lock_sha256": _sha256(s.acestep_root / "uv.lock"),
            "image_tag": s.image_tag,
            "python": sys.version,
            "torch": torch.__version__,
            "torch_cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0),
            "dtype": str(dit.dtype),
            "model_loading_seconds": time.perf_counter() - started,
            "settings": {
                "config_path": CONFIG_PATH,
                "inference_steps": s.inference_steps,
                "offload_to_cpu": s.offload_to_cpu,
                "audio_format": s.audio_format,
                "use_lm": s.use_lm,
                "lm_model": LM_MODEL if s.use_lm else None,
            },
        }

    def generate(self, job: MusicJob, out_dir: Path) -> GenerationResult:
        if self._dit is None:
            raise GenerationError("모델이 로드되지 않았습니다.")

        import soundfile as sf
        import torch
        from acestep.inference import GenerationConfig, GenerationParams, generate_music

        s = self._settings
        params = GenerationParams(
            task_type="text2music",
            caption=job.caption,
            lyrics="[Instrumental]",
            instrumental=True,
            bpm=job.bpm,
            keyscale=job.keyscale,
            timesignature=job.timesignature,
            duration=job.duration,
            seed=job.seed,
            inference_steps=s.inference_steps,
            # 문서 7장: LM은 caption 개선에만. 오디오 코드(thinking)·메타 추론은 끈다
            # 직접 준 bpm·keyscale·timesignature·duration은 LM이 덮어쓰지 않는다 (빈 값만 채움)
            thinking=False,
            use_cot_metas=False,
            use_cot_caption=self._lm is not None,
            use_cot_lyrics=False,
            use_cot_language=False,
        )
        config = GenerationConfig(
            batch_size=1,
            use_random_seed=False,
            seeds=[job.seed],
            audio_format=s.audio_format,
        )
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        started = time.perf_counter()
        try:
            with torch.inference_mode():
                result = generate_music(self._dit, self._lm, params, config, save_dir=str(out_dir))
            torch.cuda.synchronize()
        except torch.cuda.OutOfMemoryError as exc:
            torch.cuda.empty_cache()
            raise CudaOutOfMemory("CUDA 메모리 부족") from exc
        elapsed = time.perf_counter() - started

        if not result.success:
            message = str(result.error or result.status_message)
            torch.cuda.empty_cache()
            if "out of memory" in message.lower():
                raise CudaOutOfMemory(message)
            raise GenerationError(message)
        if not result.audios:
            raise GenerationError("음원 결과가 없습니다.")
        raw_path = result.audios[0].get("path")
        if not raw_path:
            raise GenerationError("추론 결과의 음원 파일 경로가 비어 있습니다.")
        path = check_audio_file(Path(raw_path).resolve())
        info = sf.info(str(path))
        peak = torch.cuda.max_memory_allocated() / 2**30
        # LM을 안 쓰면 lm_metadata가 None
        lm_caption = (result.extra_outputs.get("lm_metadata") or {}).get("caption")
        # 이전 결과 텐서가 다음 생성의 메모리를 차지하지 않게 해제
        del result
        return GenerationResult(
            path, int(info.samplerate), int(info.channels), float(info.duration),
            path.stat().st_size, elapsed, peak, lm_caption,
        )


def _load_lm(s: Settings, dtype):
    """5Hz LM (PyTorch 백엔드). 노트북 infer_music.py와 같은 초기화."""
    from acestep.llm_inference import LLMHandler

    lm = LLMHandler()
    status, success = lm.initialize(
        checkpoint_dir=str(s.model_dir),
        lm_model_path=LM_MODEL,
        backend="pt",
        device="cuda",
        offload_to_cpu=s.offload_to_cpu,
        dtype=dtype,
    )
    if not success:
        raise GenerationError(f"5Hz LM 초기화 실패: {status}")
    return lm


def make_generator(settings: Settings) -> Generator:
    if settings.generator == "fake":
        return FakeGenerator()
    return AceStepGenerator(settings)


In [ ]:
%%writefile /content/rv_src/runiverse_music/storage.py
"""사진 읽기(사진 버킷)·생성 음악 업로드(AI 버킷). 같은 키는 덮어쓴다(재요청 허용).

키 규칙 정본: Notion「S3 버킷 저장 구조」
"""
from __future__ import annotations

import shutil
from pathlib import Path
from typing import Protocol

from .config import Settings


CONTENT_TYPE = "audio/wav"


class StorageError(RuntimeError):
    """사진 읽기·결과 업로드 실패."""


def music_key(prefix: str, user_id: int, activity: str, record_card_id: int, photo_index: int | None) -> str:
    """카드 음악 객체 키. 기본 음악은 base.wav, 사진 음악은 photos/{photos 배열 순서}.wav.

    버킷 이름·s3://는 넣지 않는다 (백엔드 DB에 그대로 저장). prefix: test는 audio/test, prod는 audio.
    """
    name = "base" if photo_index is None else f"photos/{photo_index}"
    return f"{prefix.strip('/')}/{user_id}/{activity}/{record_card_id}/{name}.wav"


class Storage(Protocol):
    def read_photo(self, key: str) -> bytes: ...

    def upload(self, local_path: Path, key: str) -> None: ...


class S3Storage:
    def __init__(self, bucket: str, photo_bucket: str, client=None):
        if client is None:
            import boto3

            client = boto3.client("s3")
        self._bucket = bucket              # AI 버킷: 생성 음악
        self._photo_bucket = photo_bucket  # 기존 사진 버킷: photos[].image_key
        self._client = client

    def read_photo(self, key: str) -> bytes:
        from botocore.exceptions import BotoCoreError, ClientError

        try:
            return self._client.get_object(Bucket=self._photo_bucket, Key=key)["Body"].read()
        except (BotoCoreError, ClientError) as exc:
            raise StorageError(f"사진 읽기 실패: s3://{self._photo_bucket}/{key}") from exc

    def upload(self, local_path: Path, key: str) -> None:
        from boto3.exceptions import S3UploadFailedError
        from botocore.exceptions import BotoCoreError, ClientError

        try:
            self._client.upload_file(
                str(local_path), self._bucket, key, ExtraArgs={"ContentType": CONTENT_TYPE}
            )
        except (BotoCoreError, ClientError, S3UploadFailedError) as exc:
            raise StorageError(f"S3 업로드 실패: s3://{self._bucket}/{key}") from exc


class LocalStorage:
    def __init__(self, root: Path):
        self._root = Path(root)

    def read_photo(self, key: str) -> bytes:
        return (self._root / key).read_bytes()

    def upload(self, local_path: Path, key: str) -> None:
        target = self._root / key
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_path, target)


def make_storage(settings: Settings) -> Storage:
    if settings.storage == "s3":
        return S3Storage(settings.output_bucket, settings.photo_bucket)
    return LocalStorage(settings.local_output_dir)


In [ ]:
%%writefile /content/rv_src/runiverse_music/analyzer.py
"""Qwen3-VL 회고·사진 분석: 프롬프트 구성, 출력 검증, 실제 분석기, 테스트용 FakeAnalyzer.

분석기는 모델이 낸 원문만 돌려주고, 검증은 parse_* 함수 한곳에서 한다.
torch·transformers·PIL import는 Qwen3VLAnalyzer 메서드 안에서만 한다.
"""

import io
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Protocol

from runiverse_music import mapping_rules as rules
from runiverse_music.config import Settings

# 모델 출력이 깨졌을 때 json·dict 접근·문자열 처리에서 나는 예외
_PARSE_ERRORS = (ValueError, KeyError, TypeError, AttributeError)

QWEN_DIR = "qwen3-vl"  # model.tar.gz 안의 Qwen3-VL 폴더

# 출력은 짧은 JSON 하나라 이 정도면 충분하다. 넘으면 JSON이 잘려 parse 실패 → 기본값
MAX_NEW_TOKENS = 128
# 이미지 토큰 수는 해상도에 비례한다. 폰 원본 사진을 그대로 넣으면 T4 VRAM이 모자랄 수 있다
MAX_IMAGE_SIDE = 1024

# REVIEW_PROMPT·PHOTO_PROMPT를 바꾸면 올린다 → 결과 manifest.review_mood_classifier_version
PROMPT_VERSION = "prompt-v1"

# 프롬프트는 호출 시점에 읽는다 → Colab에서 analyzer.REVIEW_PROMPT = ... 로 바꿔 실험 가능
REVIEW_PROMPT = """러닝 회고 글을 읽고 음악 생성에 쓸 감정 단서를 뽑아라.

활동: {activity}
회고: {review_text}

규칙:
- 회고 글에 드러난 감정만 근거로 한다. 글에 없는 감정을 지어내지 않는다.
- 감정이 여러 개면 감정의 흐름과 마지막 여운을 함께 담는다.
- emotion_direction: 다음 중 정확히 하나를 그대로 쓴다 — {directions}
- emotion_cue: 회고의 감정과 여운을 담은 한글 한 구절, 30자 이내
- music_cue_en: 음악 분위기를 나타내는 짧은 영어 소문자 명사구. 장르·악기·템포는 쓰지 않는다.

아래 형식의 JSON 하나만 출력한다.
{{"emotion_direction": "...", "emotion_cue": "...", "music_cue_en": "..."}}"""

PHOTO_PROMPT = """러닝 중 찍은 사진을 보고 음악 생성에 쓸 장면 단서를 뽑아라.

활동: {activity}
사용자 설명: {user_caption}

규칙:
- 사진에 보이는 장면과 분위기를 근거로 한다. 사용자 설명은 맥락을 확인하는 데만 쓴다.
- 사진에 보이지 않는 날씨나 시간대를 단정하지 않는다.
- music_cue_en: 이 사진만의 장면·빛·공기감·움직임이 드러나는 8~15단어 영어 소문자 명사구. 다른 사진과 구별되도록 구체적으로 쓴다.
- 장르·악기·템포는 쓰지 않는다.

아래 형식의 JSON 하나만 출력한다.
{{"music_cue_en": "..."}}"""


@dataclass(frozen=True)
class ReviewAnalysis:
    emotion_direction: str  # 조성표 조회용 7종 중 하나
    emotion_cue: str        # 회고의 감정·여운 (한글). 로그에는 이 값만 남긴다
    music_cue_en: str       # 기본 음악 caption_input에 넣는 영어 단서


class Analyzer(Protocol):
    def load(self) -> None: ...

    def to_gpu(self) -> None: ...

    def to_cpu(self) -> None: ...

    def analyze_review(self, review_text: str, activity: str) -> str: ...

    def analyze_photo(self, image: bytes, user_caption: str | None, activity: str) -> str: ...


def review_messages(review_text: str, activity: str) -> list[dict]:
    """회고 분석 입력 (Qwen3-VL chat 형식, 텍스트만)."""
    text = REVIEW_PROMPT.format(
        activity=rules.ACTIVITY_CONTEXT[activity],
        review_text=review_text,
        directions=" / ".join(rules.EMOTION_DIRECTIONS),
    )
    return [{"role": "user", "content": [{"type": "text", "text": text}]}]


def photo_messages(image, user_caption: str | None, activity: str) -> list[dict]:
    """사진 분석 입력. 이미지를 텍스트보다 앞에 둔다 (Qwen3-VL 공식 예시 순서)."""
    text = PHOTO_PROMPT.format(activity=rules.ACTIVITY_CONTEXT[activity], user_caption=user_caption or "없음")
    return [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": text}]}]


def default_review() -> ReviewAnalysis:
    """회고가 없거나 해석에 실패했을 때 쓰는 담담한 기본값."""
    return ReviewAnalysis(rules.DEFAULT_EMOTION_DIRECTION, rules.DEFAULT_EMOTION_CUE, rules.DEFAULT_MUSIC_CUE_EN)


def _load_json(raw: str):
    """```json 펜스나 앞뒤 설명이 붙어도 첫 '{' ~ 마지막 '}'만 읽는다. 없으면 ValueError."""
    return json.loads(raw[raw.find("{"): raw.rfind("}") + 1])


def parse_review(raw: str) -> ReviewAnalysis | None:
    """회고 분석 원문 → ReviewAnalysis. 형식이 틀리거나 방향이 7종 밖이면 None."""
    try:
        data = _load_json(raw)
        result = ReviewAnalysis(
            emotion_direction=data["emotion_direction"],
            emotion_cue=data["emotion_cue"].strip(),
            music_cue_en=data["music_cue_en"].strip(),
        )
    except _PARSE_ERRORS:
        return None
    valid = result.emotion_direction in rules.EMOTION_DIRECTIONS and result.emotion_cue and result.music_cue_en
    return result if valid else None


def parse_photo(raw: str) -> str | None:
    """사진 분석 원문 → 사진 음악 단서(영어). 형식이 틀리거나 비어 있으면 None."""
    try:
        cue = _load_json(raw)["music_cue_en"].strip()
    except _PARSE_ERRORS:
        return None
    return cue or None


class Qwen3VLAnalyzer:
    """Qwen3-VL Instruct. 컨테이너 시작 시 1회 로드하고 요청마다 재사용한다. GPU 필수.

    평소에는 CPU에 두고 분석할 때만 GPU로 올린다 → ACE-Step 생성 중 VRAM을 차지하지 않는다.
    """

    def __init__(self, model_dir: Path | str) -> None:
        self.model_dir = model_dir  # 서빙은 폴더, 노트북은 HF 모델 ID
        self._processor = None
        self._model = None

    def load(self) -> None:
        import torch
        from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

        if not torch.cuda.is_available():
            raise RuntimeError("CUDA GPU가 필요합니다. CPU로 전환하지 않습니다.")
        self._processor = AutoProcessor.from_pretrained(self.model_dir)
        # T4는 bfloat16 연산을 지원하지 않아 float16으로 올린다
        self._model = Qwen3VLForConditionalGeneration.from_pretrained(self.model_dir, dtype=torch.float16)

    def to_gpu(self) -> None:
        self._model.to("cuda")

    def to_cpu(self) -> None:
        import torch

        self._model.to("cpu")
        torch.cuda.empty_cache()

    def analyze_review(self, review_text: str, activity: str) -> str:
        return self._generate(review_messages(review_text, activity))

    def analyze_photo(self, image: bytes, user_caption: str | None, activity: str) -> str:
        from PIL import Image

        picture = Image.open(io.BytesIO(image)).convert("RGB")
        picture.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE))  # 비율 유지 축소. 작은 사진은 그대로
        return self._generate(photo_messages(picture, user_caption, activity))

    def _generate(self, messages: list[dict]) -> str:
        import torch

        inputs = self._processor.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt",
        ).to(self._model.device)
        with torch.inference_mode():
            # do_sample=False: 같은 입력이면 같은 분석 → 재시도해도 조성이 바뀌지 않는다
            output = self._model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        new_tokens = output[:, inputs["input_ids"].shape[1]:]  # 입력 프롬프트 부분은 잘라낸다
        return self._processor.batch_decode(new_tokens, skip_special_tokens=True)[0]


class FakeAnalyzer:
    """GPU 없이 쓰는 고정 결과 (문서 예시 값). ANALYZER=fake — 로컬·테스트 전용."""

    def load(self) -> None:
        pass

    def to_gpu(self) -> None:
        pass

    def to_cpu(self) -> None:
        pass

    def analyze_review(self, review_text: str, activity: str) -> str:
        return json.dumps(
            {
                "emotion_direction": "성취·환희",
                "emotion_cue": "지친 끝에 찾아온 조용한 성취감",
                "music_cue_en": "quiet pride after a difficult finish",
            },
            ensure_ascii=False,
        )

    def analyze_photo(self, image: bytes, user_caption: str | None, activity: str) -> str:
        return json.dumps({"music_cue_en": "an open, airy coastal scene"})


def make_analyzer(settings: Settings) -> Analyzer:
    if settings.analyzer == "fake":
        return FakeAnalyzer()
    return Qwen3VLAnalyzer(settings.model_dir / QWEN_DIR)


In [ ]:
%%writefile /content/rv_src/runiverse_music/pipeline.py
"""카드 1건 처리: 전처리(①~⑥) → ACE-Step 곡별 생성 → 업로드·결과(⑦).

plan_card: 분석 → 공통 수치 → 곡별 caption_input
generate_tracks: 곡별 MusicJob → 음원
run_card: 위 전체 + 업로드 → 결과 JSON (형식: Notion「AI 기능 아키텍처 및 data 입출력」4장)
"""

import json
import logging
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Callable, Literal

from runiverse_music import mapping_rules as rules
from runiverse_music.analyzer import (
    PROMPT_VERSION, Analyzer, ReviewAnalysis, default_review, parse_photo, parse_review,
)
from runiverse_music.generator import CudaOutOfMemory, GenerationResult, Generator, MusicJob
from runiverse_music.music_params import SharedParams, decide_shared
from runiverse_music.prompting import build_caption_input, build_photo_caption
from runiverse_music.schemas import CardRequest, PhotoInput
from runiverse_music.storage import Storage, music_key

logger = logging.getLogger("runiverse_music")


@dataclass(frozen=True)
class Track:
    """곡 1개의 생성 작업. 공통 수치는 CardPlan.shared에 있고, 곡마다 다른 값만 둔다."""

    type: Literal["BASE", "PHOTO"]
    photo_index: int | None  # 요청 photos 배열 순서(0부터). 결과·파일명에 쓴다. BASE는 None
    duration: float
    caption_input: str
    image_analysis_status: Literal["SUCCEEDED", "FAILED"] | None  # PHOTO만 값이 있다


@dataclass(frozen=True)
class CardPlan:
    record_card_id: int
    mapping_version: str
    shared: SharedParams
    review_analysis: ReviewAnalysis
    tracks: list[Track]

    def to_dict(self) -> dict:
        """문서 2장 출력 형식. asdict가 안쪽 dataclass(shared 등)까지 dict로 바꾼다."""
        return asdict(self)


@dataclass(frozen=True)
class TrackResult:
    """곡 1개 생성 결과. 실패한 곡이 있어도 다른 곡은 계속 만든다."""

    track: Track
    status: Literal["SUCCEEDED", "FAILED"]
    audio: GenerationResult | None
    error: str | None


def plan_card(card: CardRequest, analyzer: Analyzer, read_photo: Callable[[str], bytes]) -> CardPlan:
    """카드 입력 → 공통 설정 + 곡별 생성 작업. 분석이 실패해도 기본값으로 이어 간다."""
    review = _analyze_review(card, analyzer)  # 조성이 emotion_direction에 의존 → 공통 수치보다 먼저
    summary = card.running_summary
    shared = decide_shared(
        avg_pace=summary.avg_pace,
        weather_code=summary.weather_code,
        emotion_direction=review.emotion_direction,
    )
    base = build_caption_input(card.activity_type, review.music_cue_en)
    tracks = [Track("BASE", None, rules.BASE_DURATION_SECONDS, base, None)]
    for index, photo in enumerate(card.photos):
        cue = _analyze_photo(card, index, photo, analyzer, read_photo)
        caption = build_photo_caption(card.activity_type, cue)  # 실패하면 cue=None → 활동 맥락만
        status = "SUCCEEDED" if cue else "FAILED"
        tracks.append(Track("PHOTO", index, rules.PHOTO_DURATION_SECONDS, caption, status))
    return CardPlan(card.record_card_id, rules.MAPPING_VERSION, shared, review, tracks)


def music_jobs(plan: CardPlan) -> list[MusicJob]:
    """곡별 ACE-Step 입력. 공통 수치는 모든 곡 동일, caption·길이·seed만 곡마다 다르다."""
    shared = plan.shared
    return [
        MusicJob(
            caption=track.caption_input,
            bpm=shared.bpm,
            keyscale=shared.keyscale,
            timesignature=shared.timesignature,
            duration=track.duration,
            # 사진 곡은 배열 순서로 결정적 파생. +1: 첫 사진이 기본 음악과 같은 seed가 되지 않게
            seed=rules.DEFAULT_SEED + (0 if track.photo_index is None else track.photo_index + 1),
        )
        for track in plan.tracks
    ]


def generate_tracks(plan: CardPlan, generator: Generator, out_dir: Path) -> list[TrackResult]:
    """곡을 순서대로 생성. 곡마다 폴더를 나눠 ACE-Step 출력 파일이 섞이지 않게 한다."""
    results = []
    for track, job in zip(plan.tracks, music_jobs(plan)):
        track_dir = out_dir / ("base" if track.photo_index is None else f"photo-{track.photo_index}")
        track_dir.mkdir(parents=True, exist_ok=True)
        audio, error, seconds = _attempt(lambda: generator.generate(job, track_dir))
        extra = {"duration": job.duration}
        if audio is not None:  # generation_seconds: 모델 생성만 / seconds: 파일 검사 포함
            extra |= {"generation_seconds": round(audio.generation_seconds, 2),
                      "peak_vram_gib": round(audio.peak_torch_allocated_gib, 2)}
        _log_stage("generation", plan.record_card_id, track.photo_index, seconds, error, **extra)
        results.append(TrackResult(track, "SUCCEEDED" if audio else "FAILED", audio, error))
    return results


def run_card(
    card: CardRequest, analyzer: Analyzer, generator: Generator, storage: Storage, output_prefix: str, work_dir: Path,
) -> dict:
    """카드 1건 전체 → 성공 결과 JSON (Notion「AI 기능 아키텍처 및 data 입출력」4.2.1, HTTP 200).

    전체 성공/실패만 있다. 실패는 CardFailure로 올린다 → app이 400·500·502 + 실패 JSON(4.2.2)으로 응답.
    한 곡이라도 생성에 실패하면 아무것도 올리지 않는다 → 같은 키의 이전 결과를 반쪽 묶음으로 덮어쓰지 않는다.
    """
    _timed("qwen_to_gpu", card.record_card_id, analyzer.to_gpu)
    plan = plan_card(card, analyzer, storage.read_photo)
    # 카드당 GPU 모델 전환 1회: 분석을 모두 끝낸 뒤 생성으로 넘어간다
    _timed("qwen_to_cpu", card.record_card_id, analyzer.to_cpu)
    results = generate_tracks(plan, generator, work_dir)

    failed = [result for result in results if result.audio is None]
    if failed:
        reasons = ", ".join(f"{_label(r.track)}({r.error})" for r in failed)
        oom = any(r.error == CudaOutOfMemory.__name__ for r in failed)
        code = "MUSIC_GENERATION_OOM" if oom else "MUSIC_GENERATION_FAILED"
        raise CardFailure(500, code, f"음악 생성 실패: {reasons}")

    items = []
    for result in results:
        track = result.track
        key = music_key(output_prefix, card.user_id, card.activity_type, card.record_card_id, track.photo_index)
        started = time.perf_counter()
        try:
            storage.upload(result.audio.audio_path, key)
        except Exception as exc:  # 업로드 하나라도 실패하면 카드 실패 (이미 올린 곡은 남는다)
            _log_stage("upload", card.record_card_id, track.photo_index, _since(started), type(exc).__name__)
            raise CardFailure(502, "MUSIC_UPLOAD_FAILED", f"음악 저장 실패: {_label(track)}({type(exc).__name__})") from exc
        item = {"s3_key": key, "duration_seconds": round(result.audio.actual_duration_seconds, 2)}
        if track.type == "PHOTO":
            # 사진 ID가 없어 배열 순서와 원본 사진 키를 함께 돌려준다 → 백엔드가 어느 사진의 음악인지 확인
            item = {"photo_index": track.photo_index, "image_key": card.photos[track.photo_index].image_key, **item}
        _log_stage("upload", card.record_card_id, track.photo_index, _since(started))
        items.append(item)

    base, *photos = items
    # 회고가 없거나 해석에 실패해 기본값을 쓴 경우는 제공할 결과가 없다 → null
    review = plan.review_analysis
    return {
        "user_id": card.user_id,
        "record_card_id": card.record_card_id,
        "status": "SUCCEEDED",
        "review_mood": None if review == default_review() else asdict(review),
        "base_music": base,
        "photo_music": photos,
        "error": None,
        "manifest": {
            "model_version": generator.manifest.get("model_revisions"),  # ACE-Step·Qwen3-VL HF 리비전
            "prompt_mapping_version": plan.mapping_version,
            "review_mood_classifier_version": PROMPT_VERSION,
        },
    }


class CardFailure(Exception):
    """카드 전체 실패. HTTP 코드와 error.code를 함께 든다 (4.2.2 표)."""

    def __init__(self, http_status: int, code: str, message: str) -> None:
        super().__init__(code)
        self.http_status, self.code, self.message = http_status, code, message


def failure_body(user_id, record_card_id, code: str, message: str, detail: list | None = None) -> dict:
    """실패 JSON (4.2.2). SageMaker가 FailureLocation에 저장한다. 일부 완성 곡은 담지 않는다.

    message에는 실패한 곡·예외 종류만 넣고 회고·캡션 원문은 넣지 않는다. 요청을 못 읽으면 ID는 None.
    """
    error = {"code": code, "message": message}
    if detail is not None:
        error["detail"] = detail
    return {"user_id": user_id, "record_card_id": record_card_id, "status": "FAILED", "error": error}


def _label(track: Track) -> str:
    return "기본 음악" if track.photo_index is None else f"사진 {track.photo_index}"


def _analyze_review(card: CardRequest, analyzer: Analyzer) -> ReviewAnalysis:
    if not card.memo:  # 회고 미작성 → 모델을 부르지 않고 기본값
        return default_review()
    review, error, seconds = _attempt(lambda: parse_review(analyzer.analyze_review(card.memo, card.activity_type)))
    _log_stage("review_analysis", card.record_card_id, None, seconds, error)
    if review is None:
        return default_review()
    return review


def _analyze_photo(card: CardRequest, index: int, photo: PhotoInput, analyzer: Analyzer, read_photo) -> str | None:
    def run() -> str | None:
        image = read_photo(photo.image_key)
        return parse_photo(analyzer.analyze_photo(image, photo.caption, card.activity_type))

    cue, error, seconds = _attempt(run)  # seconds: 사진 버킷 읽기 + Qwen 분석
    _log_stage("photo_analysis", card.record_card_id, index, seconds, error)
    return cue


def _attempt(fn):
    """(결과, 실패 사유, 걸린 초). 모델·S3·생성 예외도 그 단계의 실패로 보고 카드 처리를 이어 간다.

    예외 메시지에는 원문이 섞일 수 있어 사유는 예외 타입명만 남긴다.
    """
    started = time.perf_counter()
    try:
        result = fn()
    except Exception as exc:
        return None, type(exc).__name__, _since(started)
    return result, (None if result is not None else "invalid_output"), _since(started)


def _since(started: float) -> float:
    return round(time.perf_counter() - started, 2)


def _timed(stage: str, record_card_id: int, fn) -> None:
    """실패하면 예외를 그대로 올린다 (GPU 이동 실패는 카드 처리를 이어 갈 수 없다)."""
    started = time.perf_counter()
    fn()
    _log_stage(stage, record_card_id, None, _since(started))


def _log_stage(stage: str, record_card_id: int, photo_index: int | None, seconds: float,
               error: str | None = None, **extra) -> None:
    """단계마다 JSON 한 줄 → CloudWatch Logs. 성공 stage_done(INFO), 실패 stage_failed(WARNING).

    회고 원문·사진·사용자 캡션은 남기지 않는다 (개인정보). 숫자·식별자만.
    """
    fields = {"event": "stage_failed" if error else "stage_done", "stage": stage,
              "record_card_id": record_card_id, "photo_index": photo_index, "seconds": seconds, **extra}
    if error:
        fields["error"] = error
    logger.log(logging.WARNING if error else logging.INFO, json.dumps(fields))


In [ ]:
# 4-1. 방금 쓴 모듈을 불러옵니다. 다시 실행해도 최신 파일을 읽도록 의존 순서대로 reload합니다.
import importlib
import json
import sys

import pandas as pd
from IPython.display import display

if str(PKG.parent) not in sys.path:
    sys.path.insert(0, str(PKG.parent))

from runiverse_music import (
    analyzer, config, generator, mapping_rules as rules, music_params, pipeline, prompting, schemas, storage,
)

for module in (rules, music_params, prompting, schemas, config, generator, storage, analyzer, pipeline):
    importlib.reload(module)

pd.set_option("display.max_colwidth", None)  # 긴 caption이 "..."로 잘리지 않게
print("전처리 모듈 로드 · mapping", rules.MAPPING_VERSION)

## 4-2. 샘플 카드 (모델 없이)
문서 1장 입력 예시를 `FakeAnalyzer`(문서 5장 예시 분석값)로 처리해 문서 2장 출력 형식을 확인합니다.
- 입력 `avg_pace: 360`은 규칙표상 **160 BPM**입니다. 문서 출력 예시의 170은 5:20/km 예시 기준이라 다릅니다 (문서 8장)
- 사진 곡의 `image_analysis_status`는 문서 출력 예시에 없는 필드입니다 (사진 분석 실패 표시용)

In [ ]:
# 4-2. 문서 예시 카드 → 전처리 결과. Qwen3-VL 대신 FakeAnalyzer를 씁니다.
SAMPLE_CARD = {
    "user_id": 123,
    "record_card_id": 101,
    "memo": "힘들었지만 끝까지 달려서 뿌듯했다",
    "activity_type": "TRAVEL",
    "running_summary": {"avg_pace": 360, "weather_code": 61},
    "photos": [
        {
            "image_key": "cards/101/photos/201.jpg",
            "caption": "바닷가에서 찍은 사진",
        }
    ],
}


def show_plan(plan):
    """공통 설정·회고 분석은 JSON, 곡 목록은 표로 보여 줍니다."""
    data = plan.to_dict()
    print(json.dumps({key: data[key] for key in ("shared", "review_analysis")}, ensure_ascii=False, indent=2))
    display(pd.DataFrame(data["tracks"]))


# 규칙값 실험 예: rules.DEFAULT_BPM = 150 → 아래를 다시 실행
card = schemas.CardRequest.model_validate(SAMPLE_CARD)
show_plan(pipeline.plan_card(card, analyzer.FakeAnalyzer(), read_photo=lambda key: b""))

## 4-3. Qwen3-VL 로드
**GPU 런타임 필요.** 코랩 기본 커널에서 돌리므로 1~3번 설치 셀은 없어도 됩니다 (4-1 먼저 실행).
- `src`와 같은 `analyzer.Qwen3VLAnalyzer`를 씁니다: float16(T4는 bf16 미지원), greedy 디코딩, 사진 긴 변 1024px로 축소
- 서빙과 같은 `Qwen3-VL-2B-Instruct`를 씁니다
- `transformers>=4.57`을 설치합니다. 버전 오류가 나면 세션을 다시 시작하고 4-1부터 실행하세요

In [ ]:
# 4-3. Qwen3-VL 로드 — GPU 런타임 필요
import gc
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.57"], check=True)

import torch
import transformers
from packaging.version import Version

if Version(transformers.__version__) < Version("4.57"):
    raise RuntimeError("예전 transformers가 이미 로드돼 있습니다. 런타임 → 세션 다시 시작 후 4-1부터 다시 실행하세요.")

QWEN_MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"  # 서빙에서는 /opt/ml/model/qwen3-vl 폴더
qwen = analyzer.Qwen3VLAnalyzer(QWEN_MODEL_ID)
qwen.load()
qwen.to_gpu()  # 서빙은 run_card가 분석 전후로 올리고 내립니다
print("Qwen3-VL 로드:", QWEN_MODEL_ID, f"/ VRAM {torch.cuda.memory_allocated() / 2**30:.1f} GiB")

## 4-4. 카드 1건 실제 분석
사진을 `/content/photos/`에 `image_key`의 파일 이름(예: `201`, 확장자는 jpg·jpeg·png 무관)으로 올린 뒤 실행하세요. 없는 사진은 `FAILED`로 진행합니다.
- 확인할 것: 모델 원본 출력이 JSON으로 읽히는지, `emotion_direction` 판단, `caption_input` 문장이 자연스러운지, 피크 VRAM
- 끝에 Qwen3-VL을 GPU에서 내립니다 → 이어서 5~7번(ACE-Step)에서 이 결과(`QWEN_PLAN`)로 카드 전체 음악을 만듭니다

In [ ]:
# 4-4. 카드 1건을 실제 Qwen3-VL로 분석
PHOTO_DIR = Path("/content/photos")
PHOTO_DIR.mkdir(exist_ok=True)
QWEN_CARD = SAMPLE_CARD  # 직접 만든 카드 dict로 바꿔도 됩니다
print("사진 폴더:", sorted(path.name for path in PHOTO_DIR.iterdir()))


def read_photo(key):
    """서빙에서는 S3의 image_key를 읽을 자리. 코랩에서는 PHOTO_DIR에서 같은 이름을 찾습니다 (확장자 무관)."""
    return next(PHOTO_DIR.glob(f"{Path(key).stem}.*")).read_bytes()


class RecordingAnalyzer:
    """parse 전 모델 원본 출력을 모아 두는 래퍼."""

    def __init__(self, inner):
        self.inner, self.raw = inner, []

    def analyze_review(self, memo, activity):
        self.raw.append(("review", self.inner.analyze_review(memo, activity)))
        return self.raw[-1][1]

    def analyze_photo(self, image, caption, activity):
        self.raw.append(("photo", self.inner.analyze_photo(image, caption, activity)))
        return self.raw[-1][1]


recorder = RecordingAnalyzer(qwen)
torch.cuda.reset_peak_memory_stats()
QWEN_PLAN = pipeline.plan_card(schemas.CardRequest.model_validate(QWEN_CARD), recorder, read_photo)
print(f"피크 VRAM: {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
display(pd.DataFrame(recorder.raw, columns=["stage", "model_output"]))  # 사진 파일이 없으면 photo 행이 빠집니다
show_plan(QWEN_PLAN)

# ACE-Step(7번)이 GPU를 쓸 수 있게 Qwen3-VL을 내립니다. 다시 분석하려면 4-3부터
del qwen, recorder
gc.collect()
torch.cuda.empty_cache()

## 5. 모델 다운로드
DiT·VAE·텍스트 인코더 등의 가중치를 공식 Hugging Face 저장소에서 받습니다.
공식 다운로드 검사가 기본 1.7B LM 폴더도 검사하므로 메인 모델 묶음을 다운로드합니다. 이 1.7B LM을 `use_lm=True`일 때 caption 개선에 씁니다.

최초 실행 시 모델 저장소의 커밋도 기록합니다. 이후 같은 코랩 작업 폴더에서는 그 커밋을 재사용합니다.
모델은 실행 코드만으로 동작하지 않으며 이 다운로드가 완료되어야 합니다.

In [ ]:
download_script = ROOT / "download_models.py"
download_script.write_text("""
import json
import sys
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download

root = Path(sys.argv[1])
checkpoints = root / "ACE-Step-1.5" / "checkpoints"
checkpoints.mkdir(parents=True, exist_ok=True)
revision_file = root / "model_revisions.json"
revisions = json.loads(revision_file.read_text()) if revision_file.exists() else {}
api = HfApi()

def download(repo_id, target):
    if repo_id not in revisions:
        revisions[repo_id] = api.model_info(repo_id).sha
        revision_file.write_text(json.dumps(revisions, indent=2), encoding="utf-8")
    print(f"Downloading {repo_id} @ {revisions[repo_id]}", flush=True)
    snapshot_download(
        repo_id=repo_id,
        revision=revisions[repo_id],
        local_dir=str(target),
    )

download("ACE-Step/Ace-Step1.5", checkpoints)  # DiT·VAE·텍스트 인코더·5Hz LM 1.7B
print("모델 다운로드 완료", flush=True)
""", encoding="utf-8")
run_py311(download_script, ROOT)

## 6. 추론 스크립트 만들기
다음 셀은 `infer_music.py`를 작성합니다. 모델 로딩·생성은 4-1에서 복사한 **서빙 코드 `generator.AceStepGenerator` 그대로**입니다.

한 프로세스에서 모델을 한 번 로드한 뒤 요청 목록을 순서대로 처리합니다.
기록(`manifest.json`)에는 입력값, 5Hz LM이 개선한 caption(`lm_caption`), 실제 길이·샘플레이트, 모델 로딩 시간, 생성 시간, PyTorch GPU 메모리 피크, 코드·모델 버전이 남습니다.
메모리 피크는 해당 프로세스의 PyTorch 할당량이며 GPU 전체 사용량과는 다릅니다.

In [ ]:
INFERENCE_SCRIPT = ROOT / "infer_music.py"
INFERENCE_SCRIPT.write_text(f'''"""요청 목록을 서빙 코드(AceStepGenerator)로 순서대로 생성합니다. 인자: request.json, 결과 폴더"""
import json
import sys
from dataclasses import fields
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, {str(PKG.parent)!r})  # 4-1의 src 사본
from runiverse_music.config import load_settings
from runiverse_music.generator import AceStepGenerator, MusicJob

request_path, output = Path(sys.argv[1]), Path(sys.argv[2]).resolve()
payload = json.loads(request_path.read_text(encoding="utf-8"))
options = payload["settings"]
repo = Path(__file__).resolve().parent / "ACE-Step-1.5"
# 서빙과 같은 Settings를 코랩 경로로 만듭니다
settings = load_settings({{
    "STORAGE": "local",
    "LOCAL_OUTPUT_DIR": str(output),
    "MODEL_DIR": str(repo / "checkpoints"),
    "ACESTEP_ROOT": str(repo),
    "INFERENCE_STEPS": str(options["inference_steps"]),
    "OFFLOAD_TO_CPU": str(options["offload_to_cpu"]).lower(),
    "USE_LM": str(options["use_lm"]).lower(),
}})
generator = AceStepGenerator(settings)
generator.load()

manifest = {{"created_at_utc": datetime.now(timezone.utc).isoformat(), **generator.manifest, "runs": []}}
job_fields = [field.name for field in fields(MusicJob)]
for index, request in enumerate(payload["jobs"]):
    run_dir = output / f"run_{{index:02d}}"
    run_dir.mkdir()
    result = generator.generate(MusicJob(**{{name: request[name] for name in job_fields}}), run_dir)
    manifest["runs"].append({{
        "index": index,
        "request": request,
        "lm_caption": result.lm_caption,  # 5Hz LM이 개선한 caption (LM 미사용이면 None)
        "generation_seconds": result.generation_seconds,
        "peak_torch_allocated_gib": result.peak_torch_allocated_gib,
        "audios": [{{
            "path": str(result.audio_path),
            "relative_path": str(result.audio_path.relative_to(output)),
            "sample_rate": result.sample_rate,
            "channels": result.channels,
            "actual_duration_seconds": result.actual_duration_seconds,
            "bytes": result.size_bytes,
        }}],
    }})
    (output / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print(f"완료: run_{{index:02d}}, 생성 {{result.generation_seconds:.2f}}초", flush=True)
''', encoding="utf-8")
print("추론 스크립트 저장:", INFERENCE_SCRIPT)

## 7. 음악 생성 실행
이 셀을 다시 실행하면 새로운 결과 폴더를 만듭니다. 별도 프로세스이므로 매 실행마다 모델을 다시 로드합니다.
`generation_seconds`에는 모델 로딩·다운로드가 포함되지 않습니다. `wall_seconds`는 이번 프로세스 실행 전체 시간입니다.
**4-4를 실행했으면** `pipeline.music_jobs(QWEN_PLAN)`(서빙과 같은 함수)로 카드 전체(기본 30초 + 사진별 10초)를 만듭니다. 공통 설정(BPM·조성·박자)은 모든 곡에 같고, 사진 곡 seed는 `DEFAULT_SEED + 사진 순서 + 1`입니다. 4-4를 건너뛰면 4번 `EXPERIMENT` 1곡을 만듭니다.

In [ ]:
import time
from dataclasses import asdict
from datetime import datetime


def make_job(bpm=None, seed=None, duration=None):
    """EXPERIMENT 1곡. 조성은 비워 ACE-Step이 정합니다 (단건 API와 같음)."""
    return asdict(generator.MusicJob(
        caption=EXPERIMENT["caption"],
        bpm=EXPERIMENT["bpm"] if bpm is None else bpm,
        keyscale="",
        timesignature=rules.TIME_SIGNATURE,
        duration=EXPERIMENT["duration"] if duration is None else duration,
        seed=EXPERIMENT["seed"] if seed is None else seed,
    ))


def plan_to_jobs(plan):
    """4-4의 CardPlan → 곡별 MusicJob(dict). type·photo_index는 결과 표시용으로 붙입니다."""
    return [{**asdict(job), "type": track.type, "photo_index": track.photo_index}
            for track, job in zip(plan.tracks, pipeline.music_jobs(plan))]


def execute_jobs(jobs):
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    result_dir = OUTPUT_ROOT / stamp
    result_dir.mkdir(parents=True)
    request_file = result_dir / "request.json"
    request_file.write_text(json.dumps({"settings": EXPERIMENT, "jobs": jobs}, ensure_ascii=False, indent=2),
                            encoding="utf-8")
    started = time.perf_counter()
    try:
        run_py311(INFERENCE_SCRIPT, request_file, result_dir)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            "추론이 실패했습니다. 위 로그를 확인하세요. CUDA OOM이면 use_lm=False로 바꾸거나 "
            "길이를 10~30초로 줄인 뒤 재실행하세요. 실패 요청은 결과 폴더에 남습니다."
        ) from exc
    manifest_path = result_dir / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    manifest["wall_seconds"] = time.perf_counter() - started
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    return result_dir, manifest


# 4-4를 실행했으면 카드 전체(기본 + 사진별), 아니면 기존 실험 1곡
JOBS = plan_to_jobs(QWEN_PLAN) if "QWEN_PLAN" in globals() else [make_job()]
RESULT_DIR, MANIFEST = execute_jobs(JOBS)
print("결과:", RESULT_DIR)
print("모델 로딩:", round(MANIFEST["model_loading_seconds"], 2), "초")
print("프로세스 전체:", round(MANIFEST["wall_seconds"], 2), "초")

## 8. 결과 확인과 음악 재생
노트북 커널에서 파일을 재생합니다. 추론 라이브러리를 코랩 커널에 다시 설치할 필요는 없습니다.
목표 BPM과 실제 음원 BPM 일치 여부, 무보컬 여부, 박자 안정성은 별도로 평가하세요.
문서 7장 PoC 기록: 입력 caption · 5Hz LM이 개선한 caption · 실제 음원 → 사진별 분위기가 살아 있는지, 카드 안 곡들이 서로 어울리는지 확인

In [ ]:
from IPython.display import Audio, display

for run in MANIFEST["runs"]:
    request = run["request"]
    print(
        f"{request.get('type', 'EXPERIMENT')} {'' if request.get('photo_index') is None else request['photo_index']} / "
        f"목표 BPM={request['bpm']} / keyscale={request['keyscale'] or '-'} / seed={request['seed']} / "
        f"생성={run['generation_seconds']:.2f}초 / PyTorch 피크 할당={run['peak_torch_allocated_gib']:.2f}GiB"
    )
    print("  입력 caption:", request["caption"])
    print("  LM caption  :", run["lm_caption"])
    for audio in run["audios"]:
        print(f"  실제 길이={audio['actual_duration_seconds']:.2f}초, 샘플레이트={audio['sample_rate']}Hz, 채널={audio['channels']}")
        display(Audio(filename=audio["path"]))

## 9. 선택: 같은 모델 로딩으로 BPM 비교
첫 음원 생성이 성공한 뒤 실행하세요. 비교 셀은 기본적으로 꺼져 있습니다.
170/85 BPM은 절반·두 배 박자 해석을 비교하려는 예시이며, 특정 케이던스에 적합한 템포라는 결론은 아닙니다.



















In [ ]:
RUN_COMPARISON = False  # 비교하려면 True로 변경

if RUN_COMPARISON:
    RESULT_DIR, MANIFEST = execute_jobs([
        make_job(bpm=170, seed=42),
        make_job(bpm=85, seed=42),
        make_job(bpm=170, seed=43),
    ])
    for run in MANIFEST["runs"]:
        print(run["request"], "생성 시간:", round(run["generation_seconds"], 2), "초")
        display(Audio(filename=run["audios"][0]["path"]))
else:
    print("비교 실험을 건너뜁니다. 현재 결과 폴더:", RESULT_DIR)


## 10. 결과·코드 ZIP 다운로드
현재 `RESULT_DIR`의 음원, 입력값, 기록과 추론 스크립트를 함께 저장합니다.
`uv.lock`·코드 커밋·모델 커밋·GPU 환경을 포함합니다. 모델 가중치 자체는 용량이 커서 ZIP에 넣지 않습니다.
코랩 다운로드 권한 요청이 표시되면 허용하세요.


In [ ]:
# from google.colab import files

# shutil.copy2(INFERENCE_SCRIPT, RESULT_DIR / "infer_music.py")
# shutil.copy2(REPO / "uv.lock", RESULT_DIR / "uv.lock")
# shutil.copy2(REPO / "pyproject.toml", RESULT_DIR / "pyproject.toml")
# shutil.copy2(ROOT / "gpu_info.json", RESULT_DIR / "gpu_info.json")
# shutil.copy2(ROOT / "model_revisions.json", RESULT_DIR / "model_revisions.json")
# environment_text = subprocess.check_output(
#     [UV, "pip", "freeze", "--python", str(PY311)], cwd=REPO, text=True
# )
# (RESULT_DIR / "environment.txt").write_text(environment_text, encoding="utf-8")
# archive_path = Path(shutil.make_archive(str(RESULT_DIR), "zip", root_dir=RESULT_DIR))
# print("ZIP:", archive_path)
# files.download(str(archive_path))
import json
import shutil
import subprocess
from pathlib import Path

RESULT_DIR = Path(RESULT_DIR)
manifest_file = RESULT_DIR / "manifest.json"
if not manifest_file.is_file():
    raise FileNotFoundError(f"생성 결과 기록이 없습니다: {manifest_file}")
MANIFEST = json.loads(manifest_file.read_text(encoding="utf-8"))

# 점검·버전 기록은 보조 파일입니다. 없어도 음원 ZIP을 만들 수 있습니다.
optional_files = [
    Path(INFERENCE_SCRIPT), REPO / "uv.lock", REPO / "pyproject.toml",
    ROOT / "gpu_info.json", ROOT / "model_revisions.json",
]
missing_files = []
for source in optional_files:
    if source.is_file():
        destination = RESULT_DIR / source.name
        if source.resolve() != destination.resolve():
            shutil.copy2(source, destination)
    else:
        missing_files.append(str(source))
        print("보조 파일 없음 — 건너뜀:", source.name)

environment_error = None
try:
    environment_text = subprocess.check_output(
        [str(UV), "pip", "freeze", "--python", str(PY311)],
        cwd=REPO, text=True, stderr=subprocess.STDOUT,
    )
    (RESULT_DIR / "environment.txt").write_text(environment_text, encoding="utf-8")
except (OSError, subprocess.CalledProcessError) as exc:
    environment_error = str(exc)
    print("패키지 목록 기록 실패 — 음원 저장은 계속합니다:", exc)

(RESULT_DIR / "export_info.json").write_text(
    json.dumps({"missing_optional_files": missing_files,
                "environment_capture_error": environment_error},
               ensure_ascii=False, indent=2), encoding="utf-8",
)
archive_path = Path(shutil.make_archive(str(RESULT_DIR), "zip", root_dir=RESULT_DIR))
print("ZIP 준비 완료:", archive_path)

DOWNLOAD_ZIP = False  # 지정한 Drive 폴더에 저장합니다. PC 다운로드도 원하면 True.
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(archive_path))


## 11. 선택: Google Drive에 결과 ZIP 백업
본인이 Drive 인증을 진행하는 선택 셀입니다. 기본값은 꺼져 있습니다.
모델을 Drive에서 직접 읽도록 구성하지 않았습니다. 현재 결과 ZIP만 복사합니다.


In [ ]:
# SAVE_TO_DRIVE = False  # 백업하려면 True로 변경

# if SAVE_TO_DRIVE:
#     from google.colab import drive
#     drive.mount("/content/drive")
#     backup_dir = Path("/content/drive/MyDrive/Runiverse/ACE-Step-1.5/results")
#     backup_dir.mkdir(parents=True, exist_ok=True)
#     destination = backup_dir / archive_path.name
#     shutil.copy2(archive_path, destination)
#     print("백업 완료:", destination)
# else:
#     print("Drive 백업을 건너뜁니다.")
import hashlib
import json
import mimetypes
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

DRIVE_FOLDER_ID = "1l4inc6avgcnbPGV_xgAZb17rMDigRowr"
RESULT_DIR = Path(RESULT_DIR)
MANIFEST = json.loads((RESULT_DIR / "manifest.json").read_text(encoding="utf-8"))

# 업로드 전 음원 존재 여부를 검사합니다. ZIP 셀과 독립적으로 실행할 수 있습니다.
stamp = datetime.fromisoformat(
    MANIFEST["created_at_utc"].replace("Z", "+00:00")
).astimezone(ZoneInfo("Asia/Seoul")).strftime("%Y%m%d_%H%M%S_%f_KST")
uploads = []
for run_index, run in enumerate(MANIFEST["runs"], start=1):
    request = run["request"]
    for audio_index, audio in enumerate(run["audios"], start=1):
        # 상대 경로로 찾아서 결과 폴더를 다른 위치로 옮겨도 사용할 수 있습니다.
        source = (RESULT_DIR / audio["relative_path"] if audio.get("relative_path")
                  else Path(audio["path"]))
        if not source.is_file() or source.stat().st_size == 0:
            raise FileNotFoundError(f"음원 파일이 없거나 비어 있습니다: {source}")
        title = (
            f"Runiverse_{stamp}_BPM{int(request['bpm'])}_"
            f"{float(request['duration']):g}s_seed{int(request['seed'])}_"
            f"r{run_index:02d}_a{audio_index:02d}{source.suffix.lower()}"
        )
        uploads.append((source, title))
if not uploads:
    raise RuntimeError("저장할 음원이 없습니다. 먼저 음악 생성 셀을 실행하세요.")

uploads.append((RESULT_DIR / "manifest.json", f"Runiverse_{stamp}_manifest.json"))
current_archive = RESULT_DIR.with_suffix(".zip")
if current_archive.is_file():
    uploads.append((current_archive, f"Runiverse_{stamp}_results.zip"))

auth.authenticate_user()  # 대상 폴더에 파일을 추가할 수 있는 계정으로 인증하세요.
credentials, _ = google.auth.default()
drive_service = build("drive", "v3", credentials=credentials, cache_discovery=False)
folder = drive_service.files().get(
    fileId=DRIVE_FOLDER_ID, fields="id,name,mimeType,capabilities(canAddChildren)",
    supportsAllDrives=True,
).execute()
if folder["mimeType"] != "application/vnd.google-apps.folder":
    raise RuntimeError("저장 대상이 Drive 폴더가 아닙니다.")
if folder.get("capabilities", {}).get("canAddChildren") is False:
    raise PermissionError("현재 인증 계정에는 이 폴더에 파일을 추가할 권한이 없습니다.")
print("저장 대상 폴더:", folder["name"])

saved = []
for source, title in uploads:
    # 같은 결과로 셀을 재실행해도 동일한 파일을 중복 생성하지 않습니다.
    with source.open("rb") as content_file:
        content_md5 = hashlib.file_digest(content_file, "md5").hexdigest()
    upload_key = hashlib.sha256(
        f"{RESULT_DIR.name}/{title}".encode("utf-8")
    ).hexdigest()
    matches = drive_service.files().list(
        q=(f"'{DRIVE_FOLDER_ID}' in parents and trashed = false and "
           "appProperties has { key='runiverse_upload_key' and "
           f"value='{upload_key}' }}"),
        fields="files(id,name,parents,size,md5Checksum,webViewLink)",
        pageSize=100, supportsAllDrives=True, includeItemsFromAllDrives=True,
    ).execute().get("files", [])
    matching = next((item for item in matches
                     if item.get("md5Checksum") == content_md5), None)
    if matching is not None:
        uploaded = matching
        print("이미 저장됨:", title)
    else:
        uploaded = drive_service.files().create(
            body={"name": title, "parents": [DRIVE_FOLDER_ID],
                  "appProperties": {"runiverse_upload_key": upload_key}},
            media_body=MediaFileUpload(
                str(source), mimetype=mimetypes.guess_type(title)[0]
                or "application/octet-stream", resumable=True,
            ),
            fields="id", supportsAllDrives=True,
        ).execute(num_retries=3)
        uploaded = drive_service.files().get(
            fileId=uploaded["id"], supportsAllDrives=True,
            fields="id,name,parents,size,md5Checksum,webViewLink",
        ).execute()
        print("저장 완료:", uploaded["name"])
    if (DRIVE_FOLDER_ID not in uploaded.get("parents", [])
            or int(uploaded.get("size", -1)) != source.stat().st_size
            or uploaded.get("md5Checksum") != content_md5):
        raise RuntimeError(f"업로드한 파일의 폴더·크기·내용 확인 실패: {title}")
    if uploaded.get("webViewLink"):
        print(uploaded["webViewLink"])
    saved.append({"local_path": str(source), **uploaded})

(RESULT_DIR / "drive_uploads.json").write_text(
    json.dumps(saved, ensure_ascii=False, indent=2), encoding="utf-8",
)
print(f"총 {len(saved)}개 파일 저장 확인 완료")


In [ ]:
import zipfile
from google.colab import files

backup_path = ROOT / f"Runiverse_{stamp}_audio_backup.zip"

with zipfile.ZipFile(
    backup_path, "w", compression=zipfile.ZIP_DEFLATED
) as bundle:
    for source, title in uploads:
        if source.suffix.lower() != ".zip":
            bundle.write(source, arcname=title)

files.download(str(backup_path))

## 문제가 생겼을 때

| 상황 | 확인·조치 |
|---|---|
| GPU가 없다는 오류 | 코랩 GPU 런타임 선택 후 1번 셀부터 실행 |
| Python 버전 오류 | 2번 셀의 별도 가상환경 설치 성공 여부 확인 |
| CUDA 드라이버 오류 | 1·3번 출력과 함께 코랩 드라이버/설치된 PyTorch CUDA 호환성 확인. 시스템 CUDA나 드라이버를 임의로 덮어쓰지 않기 |
| 설치 중 디스크 부족 | 남은 디스크 확인. 모델과 PyTorch 설치 공간이 함께 필요 |
| Hugging Face 다운로드 실패 | 네트워크·저장 공간 확인 후 5번 셀 재실행 |
| CUDA out of memory | LM 끄기, 길이 10~30초, 생성 1개 유지. 다른 GPU 작업 종료 |
| 생성 성공인데 파일 없음 | 스크립트가 파일 저장까지 검사하므로 해당 오류·로그 확인 |
| 생성이 느림 | 최초 모델 로딩과 생성 시간을 구분. CPU offload와 GPU 종류 확인 |
| BPM 또는 보컬이 기대와 다름 | 실제 음원을 듣고 측정. 입력 조건은 결과 보장이 아니므로 여러 seed·설명 비교 |
| 코랩 런타임이 삭제됨 | 설치·다운로드 셀부터 재실행. 저장한 노트북·ZIP을 활용 |

**SageMaker로 옮길 때:** 추론 함수를 재사용하고 모델 로딩을 서버 시작 시 한 번 수행하도록 변경합니다. 8080 포트의 `/ping`·`/invocations`, S3 출력, 작업 상태 관리는 별도 구현 대상입니다. 이 노트북은 AWS 배포를 실행하지 않습니다.

공식 참고:
- [ACE-Step 1.5 저장소](https://github.com/ace-step/ACE-Step-1.5)
- [고정 버전 Python 추론 API](https://github.com/ace-step/ACE-Step-1.5/blob/ca1e85fe9430179831e6bc6be790c332190a3866/docs/en/INFERENCE.md)
- [고정 버전 설치 안내](https://github.com/ace-step/ACE-Step-1.5/blob/ca1e85fe9430179831e6bc6be790c332190a3866/docs/en/INSTALL.md)
- [Colab FAQ](https://research.google.com/colaboratory/faq.html)

작성일: 2026-10-06. 코드·모델의 라이선스 원문은 실제 사용하는 고정 버전에서 함께 보관하세요.
